In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.io as io
import os

# Define your device (for GPU processing)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

transform = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.Resize((128, 128)),  # Resize frames to 128x128
    transforms.ToTensor(),  # Convert frames to tensor format
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize the RGB channels
])

Device:  cuda


In [2]:
from sklearn.model_selection import train_test_split
import os

class VideoDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_frames=240, train=True, test_size=0.2):
        self.root_dir = root_dir
        self.transform = transform
        self.max_frames = max_frames
        self.classes = ['non_violent', 'violent']
        self.data = []

        # Load all data
        for label, class_name in enumerate(self.classes):
            class_dir = os.path.join(root_dir, class_name)
            for cam_folder in os.listdir(class_dir):
                cam_dir = os.path.join(class_dir, cam_folder)
                if os.path.isdir(cam_dir):
                    for video_name in os.listdir(cam_dir):
                        if video_name.endswith('.mp4'):
                            video_path = os.path.join(cam_dir, video_name)
                            self.data.append((video_path, label))

        # Split data into training and testing sets
        train_data, test_data = train_test_split(self.data, test_size=test_size, stratify=[x[1] for x in self.data])

        # Select either train or test data
        if train:
            self.data = train_data
        else:
            self.data = test_data

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        video_path, label = self.data[idx]
        
        # Load the video and process frames as you already do
        video, _, _ = io.read_video(video_path, pts_unit='sec')

        # Perform the padding or truncating of frames
        total_frames = video.shape[0]
        if total_frames < self.max_frames:
            padded_video = torch.zeros((self.max_frames, video.shape[1], video.shape[2], video.shape[3]), dtype=video.dtype)
            padded_video[:total_frames] = video
            video = padded_video
        else:
            video = video[:self.max_frames]

        # Apply transformations
        pil_transform = transforms.ToPILImage()
        frames = [self.transform(pil_transform(frame.permute(2, 0, 1))) for frame in video] if self.transform else video
        video = torch.stack(frames).permute(1, 0, 2, 3).to(device)
        label = torch.tensor(label).to(device)
        return video, label


In [3]:
path ='/home/idealab/MAIN PROJECT GROUP 15 ECA/Automatic-Violence-Detection/A-Dataset-for-Automatic-Violence-Detection-in-Videos/violence-detection-dataset'
# train_dataset = VideoDataset(root_dir=path, transform=transform)

# Create data loaders
# train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=5, shuffle=True)
train_dataset = VideoDataset(root_dir=path, transform=transform, train=True, test_size=0.2)
test_dataset = VideoDataset(root_dir=path, transform=transform, train=False, test_size=0.2)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Iterate through the data loader
i = 1
for videos, labels in train_loader:
    print(videos.shape)  # Should print shape [batch_size, frames, channels, height, width]
    print(labels)        # Should print corresponding labels (0 for non-violent, 1 for violent)
    print(i)
    i+=8
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


torch.Size([8, 3, 240, 128, 128])
tensor([1, 1, 1, 0, 1, 0, 0, 1], device='cuda:0')
1
torch.Size([8, 3, 240, 128, 128])
tensor([1, 1, 1, 1, 0, 0, 1, 1], device='cuda:0')
9
torch.Size([8, 3, 240, 128, 128])
tensor([1, 1, 0, 1, 1, 0, 1, 1], device='cuda:0')
17
torch.Size([8, 3, 240, 128, 128])
tensor([0, 1, 0, 0, 1, 1, 1, 1], device='cuda:0')
25
torch.Size([8, 3, 240, 128, 128])
tensor([1, 0, 1, 1, 1, 1, 1, 0], device='cuda:0')
33
torch.Size([8, 3, 240, 128, 128])
tensor([0, 1, 1, 0, 1, 1, 0, 0], device='cuda:0')
41
torch.Size([8, 3, 240, 128, 128])
tensor([0, 0, 0, 0, 1, 0, 0, 1], device='cuda:0')
49
torch.Size([8, 3, 240, 128, 128])
tensor([0, 1, 1, 0, 1, 1, 0, 0], device='cuda:0')
57
torch.Size([8, 3, 240, 128, 128])
tensor([1, 0, 0, 0, 0, 0, 1, 0], device='cuda:0')
65
torch.Size([8, 3, 240, 128, 128])
tensor([1, 1, 1, 1, 0, 1, 0, 1], device='cuda:0')
73
torch.Size([8, 3, 240, 128, 128])
tensor([0, 1, 1, 1, 1, 0, 1, 1], device='cuda:0')
81
torch.Size([8, 3, 240, 128, 128])
tensor([0, 

##### Total Frames: 59139
##### Average frames per video: 169
##### Time to load 350 videos: 43m 26.0s
##### VRAM used: 1190MB

In [4]:
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights

# Load the pre-trained ResNet3D-18 model with the new weights system
weights = R3D_18_Weights.DEFAULT  # Using default pre-trained weights
model = r3d_18(weights=weights)

# Modify the last fully connected layer for 2 classes (binary classification)
model.fc = nn.Linear(in_features=model.fc.in_features, out_features=2)

# Freeze all layers except the final fully connected layer
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the final fully connected layer
for param in model.fc.parameters():
    param.requires_grad = True

# Move the model to the appropriate device (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)  # Only train final layer
# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
save_dir = "checkpoints1"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# Function to save the model and optimizer state
def save_checkpoint(model, optimizer, epoch, filename="model_checkpoint.pth"):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }
    save_path = os.path.join(save_dir, filename)
    torch.save(checkpoint, save_path)
    print(f"Checkpoint saved at {save_path}")


In [6]:
def validate(model, val_loader, criterion):
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # No need to calculate gradients
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # Accumulate validation loss
            val_loss += loss.item() * inputs.size(0)
            
            # Calculate accuracy (optional)
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    
    avg_val_loss = val_loss / len(val_loader.dataset)  # Average validation loss
    val_accuracy = correct / total  # Validation accuracy (optional)
    
    return avg_val_loss, val_accuracy

In [ ]:
num_epochs = 35
train_losses = []
val_losses = []
val_accuracies = []
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for videos, labels in train_loader:
        videos, labels = videos.to(device), labels.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(videos)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * videos.size(0)
    save_checkpoint(model, optimizer, epoch, filename=f"model_epoch_{epoch}.pth")
    avg_train_loss = running_loss / len(train_loader.dataset)  # Average training loss
    train_losses.append(avg_train_loss)
    # Validation phase
    # avg_val_loss, val_accuracy = validate(model, test_loader, criterion)
    # val_losses.append(avg_val_loss)  # Store validation loss
    # val_accuracies.append(val_accuracy)  # Store validation accuracy
    
    # Print statistics
    print(f"Epoch [{epoch+1}/{num_epochs}], "
          f"Train Loss: {avg_train_loss:.4f}, ")
final_model_path = os.path.join(save_dir, "final_model.pth")
torch.save(model.state_dict(), final_model_path)
print(f"Final model saved at {final_model_path}")

In [ ]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for videos, labels in test_loader:
        videos, labels = videos.to(device), labels.to(device)
        outputs = model(videos)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy: {100 * correct / total}%')


In [ ]:
import matplotlib.pyplot as plt

# Plotting Training and Validation Loss
plt.figure(figsize=(10,5))
plt.plot(train_losses, label='Training Loss')
# plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

# Plotting Validation Accuracy
plt.figure(figsize=(10,5))
plt.plot(val_accuracies, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()
plt.show()


In [ ]:
## deleting model from memory

del model
torch.cuda.empty_cache()
del optimizer  # If you have an optimizer associated with the model
del loss_fn    # If you have a loss function using the model
gc.collect()  # Optionally trigger garbage collection